# 205. Semantic Entropy：怎样估计 LLM 语义不确定性并做拒答/升级？

> **面试问题：怎样把多次采样按语义聚类，计算 semantic entropy，避免把多解问题当幻觉，并以 risk-coverage 校准拒答阈值？**

## 先给结论

这类题要把论文概念拆为可判定的数学/状态合同：方向从什么对比样本估计、解码如何保留合法候选、熵统计的概率空间是什么、权重编辑如何验证 rewrite/generalization/locality。教学实现用受控小向量，不代表真实模型安全、语言质量或跨领域泛化。

## 一手资料

- [Semantic Entropy](https://arxiv.org/abs/2302.09664)
- [Self-Consistency](https://arxiv.org/abs/2203.11171)
- [Conformal Prediction](https://arxiv.org/abs/2107.07511)

In [ ]:
notebook_contract = {"mode": "small-controlled-arrays", "oracle": "assertions", "production": "needs-evaluation-and-versioning"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "small-controlled-arrays"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "versioning" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：表面字符串不同不等于语义不确定

多次采样可能用不同措辞表达同一答案，也可能形成相互矛盾的答案。semantic entropy 先把回答按语义等价类聚合，再在等价类概率上计算熵；它是风险信号，不是事实正确性的证明。


In [ ]:
import math  # 执行本行的状态、计算或校验逻辑。
samples = [("Paris", 0.25, "capital-paris"), ("法国首都是巴黎", 0.25, "capital-paris"), ("Lyon", 0.25, "capital-lyon"), ("巴黎", 0.25, "capital-paris")]  # 执行本行的状态、计算或校验逻辑。
assert len(samples) == 4  # 执行本行的状态、计算或校验逻辑。
assert sum(item[1] for item in samples) == 1.0  # 执行本行的状态、计算或校验逻辑。
assert {item[2] for item in samples} == {"capital-paris", "capital-lyon"}  # 执行本行的状态、计算或校验逻辑。


## 2. 聚类：语义等价判断本身也必须版本化

教学例子给出人工标签；生产可用 NLI、任务 verifier 或人工标注，但聚类器同样可能犯错。应保存 clusterer/model/version，并把不确定或不可比较回答单独处理，而不是强行合并。


In [ ]:
def cluster_mass(samples):  # 执行本行的状态、计算或校验逻辑。
    masses = {}  # 执行本行的状态、计算或校验逻辑。
    for _, probability, cluster in samples:  # 执行本行的状态、计算或校验逻辑。
        masses[cluster] = masses.get(cluster, 0.0) + probability  # 执行本行的状态、计算或校验逻辑。
    return masses  # 执行本行的状态、计算或校验逻辑。
masses = cluster_mass(samples)  # 执行本行的状态、计算或校验逻辑。
assert masses["capital-paris"] == 0.75  # 执行本行的状态、计算或校验逻辑。
assert masses["capital-lyon"] == 0.25  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(sum(masses.values()), 1.0)  # 执行本行的状态、计算或校验逻辑。


## 3. 语义熵：在 cluster mass 上计算 Shannon entropy

熵越高，模型采样的概率分布越分散；但高熵可能来自合理的多解任务，低熵也可能是自信地重复错误。因此熵只能与问题类型、证据、校准集和拒答策略结合。


In [ ]:
def entropy(probabilities):  # 执行本行的状态、计算或校验逻辑。
    return -sum(probability * math.log(probability) for probability in probabilities if probability > 0)  # 执行本行的状态、计算或校验逻辑。
semantic_entropy = entropy(masses.values())  # 执行本行的状态、计算或校验逻辑。
surface_entropy = entropy([item[1] for item in samples])  # 执行本行的状态、计算或校验逻辑。
assert semantic_entropy < surface_entropy  # 执行本行的状态、计算或校验逻辑。
assert semantic_entropy > 0  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(entropy([1.0]), 0.0)  # 执行本行的状态、计算或校验逻辑。


## 4. 决策：阈值来自校准集，不应凭直觉硬编码

一个简单 gate 是当 semantic entropy 超过经验证阈值时升级检索、请求澄清或拒答。阈值需要在标注/验证集上根据目标风险选择，并按领域、语言、任务难度与模型版本重新校准。


In [ ]:
def decide(entropy_value, threshold):  # 执行本行的状态、计算或校验逻辑。
    return "abstain_or_verify" if entropy_value > threshold else "answer_with_evidence"  # 执行本行的状态、计算或校验逻辑。
assert decide(semantic_entropy, 0.5) == "abstain_or_verify"  # 执行本行的状态、计算或校验逻辑。
assert decide(semantic_entropy, 0.7) == "answer_with_evidence"  # 执行本行的状态、计算或校验逻辑。
assert decide(0.0, 0.1) == "answer_with_evidence"  # 执行本行的状态、计算或校验逻辑。


## 5. 候选数：采样不足会低估不确定性

采样次数、temperature、top-p、随机种子和停止规则会改变观测分布。至少要记录这些因素，并检查概率质量；少量样本只能说明估计噪声，不应产生强结论。


In [ ]:
def validate_sampling(samples, expected_count):  # 执行本行的状态、计算或校验逻辑。
    return len(samples) == expected_count and math.isclose(sum(item[1] for item in samples), 1.0)  # 执行本行的状态、计算或校验逻辑。
assert validate_sampling(samples, 4)  # 执行本行的状态、计算或校验逻辑。
assert not validate_sampling(samples, 3)  # 执行本行的状态、计算或校验逻辑。
assert not validate_sampling([("x", 0.6, "x"), ("y", 0.3, "y")], 2)  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：多解、聚类错误和证据矛盾不能被熵掩盖

对于“列举两个城市”之类多解问题，高熵并非幻觉；而检索证据与回答矛盾时，即便低熵也应阻断。评测应分开标注 task ambiguity、cluster error 和 evidence conflict。


In [ ]:
def risk_gate(entropy_value, task_is_multi_answer, evidence_conflict):  # 执行本行的状态、计算或校验逻辑。
    if evidence_conflict:  # 执行本行的状态、计算或校验逻辑。
        return "block"  # 执行本行的状态、计算或校验逻辑。
    if entropy_value > 0.5 and not task_is_multi_answer:  # 执行本行的状态、计算或校验逻辑。
        return "verify"  # 执行本行的状态、计算或校验逻辑。
    return "continue"  # 执行本行的状态、计算或校验逻辑。
assert risk_gate(0.1, False, True) == "block"  # 执行本行的状态、计算或校验逻辑。
assert risk_gate(0.6, False, False) == "verify"  # 执行本行的状态、计算或校验逻辑。
assert risk_gate(0.6, True, False) == "continue"  # 执行本行的状态、计算或校验逻辑。


## 7. 评测：风险—覆盖曲线比单点准确率更有信息

若系统会拒答或升级，必须同时报告回答覆盖率和被回答样本的错误率，形成 risk-coverage 曲线。示例计算两个阈值下的覆盖和错误；生产应有置信区间、分组切片和漂移监控。


In [ ]:
def risk_coverage(rows, threshold):  # 执行本行的状态、计算或校验逻辑。
    answered = [row for row in rows if row["entropy"] <= threshold]  # 执行本行的状态、计算或校验逻辑。
    coverage = len(answered) / len(rows)  # 执行本行的状态、计算或校验逻辑。
    risk = sum(not row["correct"] for row in answered) / max(1, len(answered))  # 执行本行的状态、计算或校验逻辑。
    return coverage, risk  # 执行本行的状态、计算或校验逻辑。
rows = [{"entropy": 0.1, "correct": True}, {"entropy": 0.6, "correct": False}]  # 执行本行的状态、计算或校验逻辑。
assert risk_coverage(rows, 0.2) == (0.5, 0.0)  # 执行本行的状态、计算或校验逻辑。
assert risk_coverage(rows, 1.0) == (1.0, 0.5)  # 执行本行的状态、计算或校验逻辑。
assert risk_coverage(rows, 0.2)[0] < risk_coverage(rows, 1.0)[0]  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：采样、聚类、阈值和证据版本共同决定结论

semantic entropy 的数值脱离采样配置和 clusterer 没有可比性。制品应绑定模型、decode 参数、样本数、聚类器、阈值和证据索引版本，才能重放拒答/升级决策。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"model": "demo-v1", "samples": 4, "clusterer": "manual-v1", "threshold": 0.5, "index": "facts-v1"}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["samples"] == len(samples)  # 执行本行的状态、计算或校验逻辑。
assert artifact["threshold"] == 0.5  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时先说明目标与状态，再给出核心公式、失败反例、独立评测和版本化制品。不要把对一个合成向量/几个候选的断言通过，误说成真实大模型上已经可靠、无偏或安全。
